<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/03_select_hyperparam_for_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [1]:
%pip install torchinfo -qq
%pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
%pip install starfile -qq
%pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.1 MB/s eta 0:00:00
     - 10.7 MB 10.1 MB/s 0:00:01
  Preparing metadata (setup.py) .

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [2]:
%pip install pycuda==2024.1
%pip install "numpy<2.0"
%pip install mrcfile -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.6/105.6 kB 9.8 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2024.1-cp311-cp311-linux_x86_64.whl size=659519 sha256=66368cd9ae5ad777785a5a85b0b4ac7940714edd8630e21c79a74b452a8510c6
  Stored in directory: /root/.cache/pip/wheels/9f/8c/1d/f9e11f5c0a8f91fd93b22ebf74c364ec59bc28bb2f76b2f64d
Successfully built pycuda
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 110.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.5 MB/s eta 0:00:00


## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [1]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/research_xc/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
LABEL_DIR = "/content/drive/MyDrive/research_xc/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/research_xc/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/research_xc/results/10017_test/unet_eb5_dice_CDCRF" # @param {type:"string"}

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rm: cannot remove '/content/sample_data': No such file or directory


In [3]:
IMAGE_DIR = "/content/image_dir"

In [5]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

Cloning into 'CryoParticleSegment'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 190 (delta 86), reused 27 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 29.46 MiB | 12.67 MiB/s, done.
Resolving deltas: 100% (86/86), done.


In [4]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

### ✅ Packages Handling

In [5]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

In [6]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery

## ⭐ Main

### ✅ Setting

In [7]:
# @markdown Parameters.

user = True # @param {type:"boolean"}

In [8]:
# @markdown Parameters.

BATCH = 2
CROP_SIZE = (512, 512)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

### ✅ Dataset

You can provide a [`transforms.CenterCrop(3840)`](https://docs.pytorch.org/vision/master/generated/torchvision.transforms.CenterCrop.html) object to crop out boundary artifacts.


In [10]:
crop = transforms.CenterCrop(3840)

In [11]:
train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)

In [12]:
val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True)

In [13]:
if not user:
    test_dir = os.path.join(IMAGE_DIR, 'test')
    test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, filenames=test_filenames, crop_size=CROP_SIZE, crop=crop)
    test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True)

In [14]:
for i1, i2, i3, i4 in val_loader: #test loader and reconstruct
    print(i2.dtype, i4.dtype)
    print(i2.shape, i4.shape)
    shape = i4.shape
    break

torch.int64 torch.int64
torch.Size([81, 1, 512, 512]) torch.Size([1, 3840, 3840])


## ⭐ Evaluate

In [15]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
import starfile
import pandas as pd
import matplotlib
from PIL import Image
import cv2

def get_basename_with_uid_removed(path):
  return os.path.basename(path).split(sep='_', maxsplit=1)[-1]


def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy


In [16]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}
SIZE = 4096 # @param {type:"integer"}

In [17]:
!cp {DATASET_DIR}/{EMPIAR_ID}/filtered_val.star .

In [18]:
y_size = SIZE
labeled_particles = starfile.read(f"filtered_val.star")['particles']
labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']
labeled_particles

,image_name,x_coord,y_coord
0,Falcon_2012_06_12-16_59_12_0,1876,3762
1,Falcon_2012_06_12-16_59_12_0,1719,920
2,Falcon_2012_06_12-16_59_12_0,3171,718
3,Falcon_2012_06_12-16_59_12_0,3772,3782
4,Falcon_2012_06_12-16_59_12_0,1492,2245
...,...,...,...
4270,Falcon_2012_06_13-01_56_06_0,2205,491
4271,Falcon_2012_06_13-01_56_06_0,465,3216
4272,Falcon_2012_06_13-01_56_06_0,2685,1436
4273,Falcon_2012_06_13-01_56_06_0,1072,2705


In [19]:
def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

def plot_micrograph_and_labels(ax, micrograph, labels, coords):
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(labels, cmap='gray', alpha=0.5)
    for x, y in coords:
        corrected_x, corrected_y = x, y
        circle = matplotlib.patches.Circle((corrected_x, corrected_y), radius=RADIUS, fill=False, color='r')
        ax.add_patch(circle)

You can specify a `crop_size` in `preprocess_and_crop()` to remove boundary artifacts during preprocessing.

In [23]:
label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)
gts = []

for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    # if idx == 6:
    #     break
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)
    print(cropped_label_image.shape)
    label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    coords = locations[['x_coord', 'y_coord']].values - BORDER
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, coords)
    plt.show()
    print(len(coords))
    gts.append(coords)
    ##

#filename = f"{os.path.splitext(checkpoint_path)[0]}.png"
#pred_path = os.path.join(RESULT_DIR, "Each_ckpt", filename)
#save_image(pred_image, pred_path)

Output hidden; open in https://colab.research.google.com to view.

In [24]:
label_images.shape

(6, 3840, 3840)

### CV approach

In [25]:
radius = RADIUS

In [26]:
from center_finding import normalize, min_rect_circle, eliminate_near

In [27]:
cv_list_all = []
cv_config = []
e_factor = [2,4,6]
s_factor = [0.6,1,1.4]

for e in e_factor:
    for s in s_factor:
        cv_list = []
        print(f"e_factor: {e}, s_factor: {s}")
        for img in label_images:
            thresh1 = normalize(img)
            kernel_size = int(radius / 4)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            kernel_size = int(radius / e)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cont_array = [c for c in contours]

            # Filter out small/large region and find bounding box with center
            contr_min = radius**s
            c_ = np.array([cv2.contourArea(contour) for contour in contours])
            aa = (c_>contr_min) & (c_<500000)
            aa = aa.tolist()
            c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
            c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
            c_list = [x for x in c_list if x is not None]
            cv_list.append(c_list)
            print(len(c_), len(c_list))
        cv_list_all.append(cv_list)
        cv_config.append((e,s))

e_factor: 2, s_factor: 0.6
692 520
714 618
724 602
719 600
620 525
841 686
e_factor: 2, s_factor: 1
692 406
714 513
724 492
719 499
620 459
841 558
e_factor: 2, s_factor: 1.4
692 120
714 177
724 169
719 178
620 238
841 236
e_factor: 4, s_factor: 0.6
607 598
619 604
604 593
621 604
576 575
691 669
e_factor: 4, s_factor: 1
607 598
619 604
604 593
621 604
576 575
691 669
e_factor: 4, s_factor: 1.4
607 598
619 604
604 593
621 604
576 575
691 669
e_factor: 6, s_factor: 0.6
580 563
581 548
573 545
583 550
549 537
652 613
e_factor: 6, s_factor: 1
580 563
581 548
573 545
583 550
549 537
652 613
e_factor: 6, s_factor: 1.4
580 563
581 548
573 545
583 550
549 537
652 613


In [28]:
observe_id = 0 # @param {type:"integer"}

In [29]:
for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    #locations = labeled_particles[labeled_particles['image_name'] == name]
    #adjusted_c_list = [(x + 128, y + 128) for x, y in cv_list_all[0][idx]]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, cv_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### TrackPy

More information about the Crocker–Grier centroid-finding algorithm is available in the [TrackPy documentation](https://soft-matter.github.io/trackpy/dev/generated/trackpy.locate.html).


In [30]:
import trackpy as tp

In [31]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = [0.4, 0.7, 1]
#topn = 500
scale = [0.15, 0.25, 0.35]  # Scale factor (0.5 means reducing the size to half)

In [32]:
label_images.shape

(6, 3840, 3840)

In [33]:
tp_list_all = []
tp_config = []
for e in scale:
    for s in sep:
        print(f"e_factor: {e}, s_factor: {s}")
        tp_list = []
        for img in label_images:
            output_image = img
            small_image = cv2.resize(output_image, None, fx=e, fy=e, interpolation=cv2.INTER_AREA)
            # Adjust parameters based on the scale
            sep_s = round(s * TrackParticleSize)+1
            small_sep = int(sep_s * e)
            small_diameter = int(TrackParticleSize * e)
            # Ensure small_diameter is odd
            if small_diameter % 2 == 0:
                small_diameter += 1
            coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
            #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
            coorTrack['x'] *= (1/e)
            coorTrack['y'] *= (1/e)
            coords = coorTrack[['x', 'y']].values
            tp_list.append(coords)
            print(len(coords))
        tp_list_all.append(tp_list)
        tp_config.append((e,s))

e_factor: 0.15, s_factor: 0.4
588
596
611
603
551
694
e_factor: 0.15, s_factor: 0.7
524
504
524
507
483
569
e_factor: 0.15, s_factor: 1
316
277
283
298
298
318
e_factor: 0.25, s_factor: 0.4
574
593
599
588
544
676
e_factor: 0.25, s_factor: 0.7
504
490
511
487
475
539
e_factor: 0.25, s_factor: 1
291
273
263
265
272
298
e_factor: 0.35, s_factor: 0.4
559
581
595
575
535
658
e_factor: 0.35, s_factor: 0.7
492
480
502
484
464
531
e_factor: 0.35, s_factor: 1
299
280
281
265
280
294


In [34]:
observe_id = 3 # @param {type:"integer"}

In [35]:
for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, tp_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Nonmax

In [36]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [37]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more 2000 for 10017, should inverse proportional to pCans. Use 1000 for 10081
pCans=[0.4, 0.5, 0.6] #the grid size smaller will generate more candidate
overlaps=[0, 0.3, 0.6] #allow for overlap, smaller will delete more
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [38]:
nms_list_all = []
nms_config = []
for e in pCans:
    for s in overlaps:
        pCan=e
        overlap=s
        print(f"e_factor: {e}, s_factor: {s}")
        nms_list = []
        for img in label_images:
            heatArr=normalize(pad_image(img))
            heatArr=reNorm(heatArr, nSep)
            heatArr=reLev(heatArr,level)
            gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

            # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
            gsize=psize*pCan
            heat=heatArr.astype(np.float32)
            gaus=gaus.astype(np.float32)
            score=np.zeros(heat.shape).astype(np.float32)
            [sizex,sizey]=heat.shape
            sizex = int(sizex)
            sizey = int(sizey)
            psize = int(psize)
            gsize = int(gsize)

            func = scoreGpu

            tx=16
            ty=16
            bx=(sizex-1)//tx+1
            by=(sizey-1)//ty+1
            print('get score gpu',tx, ty, bx, by, gsize, psize)


            func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
                block=(tx, ty, 1), grid=(int(bx), int(by)))

            # # Calculate necessary dimensions and convert all to int
            numx = int((sizex - 1) // gsize + 1)
            numy = int((sizey - 1) // gsize + 1)
            num = numx * numy
            tnum = num * 5

            # # Create the result array
            res = np.zeros(tnum).astype(np.float32)

            # # Block and grid dimensions
            bx = (numx - 1) // tx + 1
            by = (numy - 1) // ty + 1

            # Get the function from the module
            func = getMax

            # Call the function, ensure all parameters are the correct type
            func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
                block=(16, 16, 1), grid=(bx, by, 1))

            # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
            niter = np.int32(nIter)
            gsize = np.int32(gsize)
            sizex = np.int32(sizex)
            sizey = np.int32(sizey)
            numx = np.int32(numx)
            tnum = np.int32(num)

            print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
            func = getMax3
            # Ensuring the correct parameter order and type for the kernel invocation
            func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
                    block=(16, 16, 1), grid=(bx, by, 1))

            canList=reshape(res,num)

            print('Number of Particles before:', len(canList))
            if(len(canList)>pnum):
                canList=canList[:pnum]


            canList=cleanCanList(canList,overlap,psize)
            #canList=reCan(canList,ratio)
            print('Number of Particles:', len(canList))
            nms_list.append([(r[1],r[0]) for r in canList])
        nms_list_all.append(nms_list)
        nms_config.append((e,s))
        print("\n")

e_factor: 0.4, s_factor: 0
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5492
Number of Particles: 277
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5430
Number of Particles: 268
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5475
Number of Particles: 251
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5460
Number of Particles: 253
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5407
Number of Particles: 252
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5503
Number of Particles: 283


e_factor: 0.4, s_factor: 0.3
get score gpu 16 1

In [39]:
observe_id = 2 # @param {type:"integer"}

In [40]:
for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, nms_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Score

> #### 🗒 Info
> Here, we compute the score based on the validation set. You may choose to rank the algorithms using either the F-score or mAP. Additionally, the parameter $\beta$ in the F-score can be adjusted: values of $\beta > 1$ place greater emphasis on recall over precision, while $\beta < 1$ give more weight to precision.


In [41]:
from metrics import centers_to_boxes, calculate_iou_torchvision, evaluate_detection_raw_multiple, f_beta_score, calculate_mAP_multiple_images

In [42]:
# Assign a default confidence score of 1.0 to all predicted boxes
default_score = 1.0
beta = 1 # @param {type:"number"}
F_score = False # @param {type:"boolean"}

In [43]:
width = RADIUS*2
height = width
cv_scores = []
for i, cv_list in enumerate(cv_list_all):
    print(f"config {cv_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(cv_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    cv_scores.append((f_beta, mAP_value.item()))

config (2, 0.6)
Precision: 0.9880268573760986
Recall: 0.8207336068153381
F_Beta: 0.8966437096989575
Mean Average Precision (mAP): 0.6892116069793701
config (2, 1)
Precision: 0.9971134066581726
Recall: 0.6834819316864014
F_Beta: 0.8110328307401334
Mean Average Precision (mAP): 0.574160635471344
config (2, 1.4)
Precision: 0.9922029376029968
Recall: 0.26015353202819824
F_Beta: 0.4122230450603071
Mean Average Precision (mAP): 0.20654863119125366
config (4, 0.6)
Precision: 0.962712824344635
Recall: 0.8239333033561707
F_Beta: 0.8879331449551114
Mean Average Precision (mAP): 0.6801626086235046
config (4, 1)
Precision: 0.962712824344635
Recall: 0.8239333033561707
F_Beta: 0.8879331449551114
Mean Average Precision (mAP): 0.6801626086235046
config (4, 1.4)
Precision: 0.962712824344635
Recall: 0.8239333033561707
F_Beta: 0.8879331449551114
Mean Average Precision (mAP): 0.6801626086235046
config (6, 0.6)
Precision: 0.9466004967689514
Recall: 0.7468063235282898
F_Beta: 0.8349171957604228
Mean Average

In [44]:
width = RADIUS*2
height = width
tp_scores = []

for i, tp_list in enumerate(tp_list_all):
    print(f"config {tp_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(tp_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    tp_scores.append((f_beta, mAP_value.item()))

config (0.15, 0.4)
Precision: 0.9992005228996277
Recall: 0.8532168865203857
F_Beta: 0.920456431496061
Mean Average Precision (mAP): 0.6533084511756897
config (0.15, 0.7)
Precision: 1.0
Recall: 0.7307845950126648
F_Beta: 0.8444547023569011
Mean Average Precision (mAP): 0.5710253715515137
config (0.15, 1)
Precision: 1.0
Recall: 0.4216013252735138
F_Beta: 0.5931358078783422
Mean Average Precision (mAP): 0.3369666039943695
config (0.25, 0.4)
Precision: 0.9988877773284912
Recall: 0.8370216488838196
F_Beta: 0.9108191095835672
Mean Average Precision (mAP): 0.684084415435791
config (0.25, 0.7)
Precision: 0.9996907711029053
Recall: 0.7065398693084717
F_Beta: 0.8279318985780679
Mean Average Precision (mAP): 0.5890331864356995
config (0.25, 1)
Precision: 1.0
Recall: 0.39118871092796326
F_Beta: 0.5623805136645036
Mean Average Precision (mAP): 0.33492350578308105
config (0.35, 0.4)
Precision: 0.998538076877594
Recall: 0.8202807307243347
F_Beta: 0.9006741517448515
Mean Average Precision (mAP): 0.645

In [45]:
width = RADIUS*2
height = width
nms_scores = []

for i, nms_list in enumerate(nms_list_all):
    print(f"config {nms_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(nms_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    nms_scores.append((f_beta, mAP_value.item()))

config (0.4, 0)
Precision: 0.9738032817840576
Recall: 0.3630460798740387
F_Beta: 0.5289084532032621
Mean Average Precision (mAP): 0.1776331514120102
config (0.4, 0.3)
Precision: 0.9967041015625
Recall: 0.6439552903175354
F_Beta: 0.7824084417020649
Mean Average Precision (mAP): 0.4814085066318512
config (0.4, 0.6)
Precision: 0.9976906180381775
Recall: 0.7992017269134521
F_Beta: 0.887483401108178
Mean Average Precision (mAP): 0.6545977592468262
config (0.5, 0)
Precision: 0.9773961901664734
Recall: 0.4028981029987335
F_Beta: 0.5705900152542742
Mean Average Precision (mAP): 0.1886347085237503
config (0.5, 0.3)
Precision: 0.9960866570472717
Recall: 0.77603679895401
F_Beta: 0.8723996042126835
Mean Average Precision (mAP): 0.584636390209198
config (0.5, 0.6)
Precision: 0.994929850101471
Recall: 0.92446368932724
F_Beta: 0.9584032674407837
Mean Average Precision (mAP): 0.7522952556610107
config (0.6, 0)
Precision: 0.9717208743095398
Recall: 0.42480286955833435
F_Beta: 0.5911676298079187
Mean Av

### Write the best hyperparameters

In [46]:
if F_score:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[0], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][0], reverse=True)
else:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[1], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][1], reverse=True)
cv_scores_sorted[0], cv_config[csorted_indices[0]]

((0.8966437096989575, 0.6892116069793701), (2, 0.6))

In [47]:
if F_score:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[0], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][0], reverse=True)
else:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[1], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][1], reverse=True)
tp_scores_sorted[0], tp_config[tsorted_indices[0]]

((0.9108191095835672, 0.684084415435791), (0.25, 0.4))

In [48]:
if F_score:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[0], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][0], reverse=True)
else:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[1], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][1], reverse=True)
nms_scores_sorted[0], nms_config[nsorted_indices[0]]

((0.9584032674407837, 0.7522952556610107), (0.5, 0.6))

In [49]:
with open("best_config.txt", "w") as f:
    f.write(f"cv_config: {cv_config[csorted_indices[0]]}\n")
    f.write(f"tp_config: {tp_config[tsorted_indices[0]]}\n")
    f.write(f"nms_config: {nms_config[nsorted_indices[0]]}\n")

In [50]:
!cp best_config.txt {RESULT_DIR}